In [21]:
import sys
import importlib
import datetime
import io
import urllib.request

import pandas as pd
import pyarrow.parquet as pq

sys.path.insert(0, '/Users/tanishbhilare/Desktop/mbta-performance-analyzer')

In [22]:
target = (datetime.date.today() - datetime.timedelta(days=14)).strftime("%Y-%m-%d")
url = (
    f"https://performancedata.mbta.com/lamp/subway-on-time-performance-v1/"
    f"{target}-subway-on-time-performance-v1.parquet"
)
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=60) as r:
    data = r.read()

df_schema = pq.read_table(io.BytesIO(data)).to_pandas()
print(f"Schema sample — {target}")
print(f"Shape: {df_schema.shape}")
df_schema.head(3)

Schema sample — 2026-03-04
Shape: (41572, 27)


,stop_sequence,stop_id,parent_station,move_timestamp,stop_timestamp,travel_time_seconds,dwell_time_seconds,headway_trunk_seconds,headway_branch_seconds,service_date,...,trip_id,vehicle_label,vehicle_consist,direction,direction_destination,scheduled_arrival_time,scheduled_departure_time,scheduled_travel_time,scheduled_headway_branch,scheduled_headway_trunk
0,1,Alewife-02,place-alfcl,NaN,1.772606e+09,NaN,NaN,174.0,NaN,20260304,...,75229666,1849,1849|1848|1802|1803|1832|1833,South,Ashmont/Braintree,19320.0,19320.0,NaN,NaN,420.0
1,1,70106,place-lake,NaN,1.772615e+09,NaN,NaN,NaN,NaN,20260304,...,ADDED-1583539796,3836-3665,3836|3665,East,Government Center,32040.0,32040.0,NaN,480.0,480.0
2,310,70107,place-lake,NaN,1.772615e+09,NaN,NaN,NaN,NaN,20260304,...,ADDED-1583539797,3618,3618,West,Boston College,85980.0,85980.0,180.0,600.0,600.0


In [23]:
import utils.lamp as lamp
importlib.reload(lamp)

df = lamp.load_lamp_days(days_back=30)

Loading 31 days of LAMP data (2026-02-13 to 2026-03-15)...
  ✓ 2026-02-13 — 45,093 rows
  ✓ 2026-02-14 — 35,316 rows
  ✓ 2026-02-15 — 32,928 rows
  ✓ 2026-02-16 — 34,461 rows
  ✓ 2026-02-17 — 46,374 rows
  ✓ 2026-02-18 — 43,906 rows
  ✓ 2026-02-19 — 45,558 rows
  ✓ 2026-02-20 — 46,045 rows
  ✓ 2026-02-21 — 35,265 rows
  ✓ 2026-02-22 — 31,973 rows
  ✓ 2026-02-23 — 21,614 rows
  ✓ 2026-02-24 — 34,800 rows
  ✓ 2026-02-25 — 40,579 rows
  ✓ 2026-02-26 — 44,974 rows
  ✓ 2026-02-27 — 45,441 rows
  ✓ 2026-02-28 — 33,985 rows
  ✓ 2026-03-01 — 32,358 rows
  ✓ 2026-03-02 — 37,110 rows
  ✓ 2026-03-03 — 40,530 rows
  ✓ 2026-03-04 — 39,468 rows
  ✓ 2026-03-05 — 40,080 rows
  ✓ 2026-03-06 — 39,947 rows
  ✓ 2026-03-07 — 37,159 rows
  ✓ 2026-03-08 — 33,882 rows
  ✓ 2026-03-09 — 44,694 rows
  ✓ 2026-03-10 — 44,088 rows
  ✓ 2026-03-11 — 46,812 rows
  ✓ 2026-03-12 — 45,235 rows
  ✓ 2026-03-13 — 47,152 rows
  ✓ 2026-03-14 — 36,114 rows
  ✓ 2026-03-15 — 35,173 rows

Loaded: 1,218,114 rows across 31 days
Lin

In [24]:
print("date dtype:    ", df["date"].dtype)
print("date range:    ", df["date"].min().date(), "to", df["date"].max().date())
print("lines:         ", sorted(df["line"].unique()))
print("period counts:\n", df["period"].value_counts())
print("is_on_time:    ", df["is_on_time"].value_counts(dropna=False).to_dict())
print("ett_min nulls: ", df["ett_min"].isna().sum())
print("total rows:    ", len(df))

date dtype:     datetime64[ns]
date range:     2026-02-13 to 2026-03-15
lines:          ['Blue', 'Green', 'Orange', 'Red']
period counts:
 period
Off-Peak    857910
Peak        360204
Name: count, dtype: int64
is_on_time:     {True: 1118869, nan: 94451, False: 4794}
ett_min nulls:  94451
total rows:     1218114


In [25]:
print("=== OTP by stop — Red (worst 5) ===")
print(m.otp_by_stop(df, 'Red').head().to_string(index=False))

print("\n=== OTP by stop — Green (worst 5) ===")
print(m.otp_by_stop(df, 'Green').head().to_string(index=False))

print("\n=== Headway by stop — Red (worst 5) ===")
print(m.headway_by_stop(df, 'Red').head().to_string(index=False))

print("\n=== OTP trend (first 5 rows) ===")
print(m.otp_trend(df).head().to_string(index=False))

print("\n=== ETT trend (first 5 rows) ===")
print(m.ett_trend(df).head().to_string(index=False))

=== OTP by stop — Red (worst 5) ===
parent_station  on_time  total_trips  otp_pct
   place-smmnl   5035.0         5326     94.5
   place-nqncy   5423.0         5589     97.0
     place-jfk  10211.0        10455     97.7
   place-shmnl   5232.0         5297     98.8
   place-andrw  10421.0        10500     99.2

=== OTP by stop — Green (worst 5) ===
parent_station  on_time  total_trips  otp_pct
   place-mdftf    338.0          349     96.8
   place-unsqu     44.0           45     97.8
   place-boyls  27358.0        27934     97.9
   place-balsq   7715.0         7825     98.6
   place-esomr   7385.0         7462     99.0

=== Headway by stop — Red (worst 5) ===
parent_station  avg_actual_min  avg_scheduled_min  headway_ratio_pct
   place-qnctr            12.9               11.8              108.9
   place-wlsta            12.8               11.8              108.5
   place-nqncy            12.8               11.8              108.5
   place-qamnl            12.8               11.8       

In [26]:
print("=== OTP by stop — Red (worst 5) ===")
print(m.otp_by_stop(df, 'Red').head().to_string(index=False))

print("\n=== OTP by stop — Green (worst 5) ===")
print(m.otp_by_stop(df, 'Green').head().to_string(index=False))

print("\n=== Headway by stop — Red (worst 5) ===")
print(m.headway_by_stop(df, 'Red').head().to_string(index=False))

print("\n=== OTP trend (first 5 rows) ===")
print(m.otp_trend(df).head().to_string(index=False))

print("\n=== ETT trend (first 5 rows) ===")
print(m.ett_trend(df).head().to_string(index=False))

=== OTP by stop — Red (worst 5) ===
parent_station  on_time  total_trips  otp_pct
   place-smmnl   5035.0         5326     94.5
   place-nqncy   5423.0         5589     97.0
     place-jfk  10211.0        10455     97.7
   place-shmnl   5232.0         5297     98.8
   place-andrw  10421.0        10500     99.2

=== OTP by stop — Green (worst 5) ===
parent_station  on_time  total_trips  otp_pct
   place-mdftf    338.0          349     96.8
   place-unsqu     44.0           45     97.8
   place-boyls  27358.0        27934     97.9
   place-balsq   7715.0         7825     98.6
   place-esomr   7385.0         7462     99.0

=== Headway by stop — Red (worst 5) ===
parent_station  avg_actual_min  avg_scheduled_min  headway_ratio_pct
   place-qnctr            12.9               11.8              108.9
   place-wlsta            12.8               11.8              108.5
   place-nqncy            12.8               11.8              108.5
   place-qamnl            12.8               11.8       

In [1]:
import requests

resp = requests.get(
    "https://api-v3.mbta.com/vehicles",
    params={
        "filter[route]": "Red",
        "filter[direction_id]": 1,
        "fields[vehicle]": "label,current_status,latitude,longitude,speed",
    },
    timeout=15
)

print("Status:", resp.status_code)
data = resp.json()
vehicles = data.get("data", [])
print(f"Active Red Line inbound vehicles: {len(vehicles)}")
if vehicles:
    v = vehicles[0]["attributes"]
    print(f"Sample: label={vehicles[0]['attributes'].get('label')}, status={v.get('current_status')}, lat={v.get('latitude')}")

Status: 200
Active Red Line inbound vehicles: 11
Sample: label=1835, status=IN_TRANSIT_TO, lat=42.36953


In [2]:
# Test predictions with schedule included
resp2 = requests.get(
    "https://api-v3.mbta.com/predictions",
    params={
        "filter[route]": "Red",
        "filter[direction_id]": 1,
        "include": "stop,schedule",
        "fields[prediction]": "departure_time,arrival_time,status",
        "fields[stop]": "name",
        "fields[schedule]": "departure_time,arrival_time",
        "page[limit]": 10,
    },
    timeout=15
)
print("Predictions status:", resp2.status_code)
preds = resp2.json()
print(f"Predictions returned: {len(preds.get('data', []))}")
print(f"Included resources: {len(preds.get('included', []))}")

# Spot check one prediction
if preds.get("data"):
    d = preds["data"][0]
    print(f"\nSample prediction:")
    print(f"  attributes: {d['attributes']}")
    print(f"  relationships keys: {list(d['relationships'].keys())}")

# Test alerts
resp3 = requests.get(
    "https://api-v3.mbta.com/alerts",
    params={
        "filter[route]": "Red",
        "filter[activity]": "BOARD,EXIT,RIDE",
        "fields[alert]": "header,effect,severity,updated_at,lifecycle",
        "page[limit]": 5,
    },
    timeout=15
)
print(f"\nAlerts status: {resp3.status_code}")
print(f"Active alerts: {len(resp3.json().get('data', []))}")

Predictions status: 200
Predictions returned: 10
Included resources: 12

Sample prediction:
  attributes: {'arrival_time': None, 'departure_time': '2026-03-18T19:07:07-04:00', 'status': None}
  relationships keys: ['route', 'schedule', 'stop', 'trip', 'vehicle']

Alerts status: 200
Active alerts: 1


In [4]:
# Cell 7 — test api.py
import utils.api as api
importlib.reload(api)

print("=== Vehicles — Red Line inbound ===")
veh = api.get_vehicles("Red", 1)
print(f"Active vehicles: {len(veh)}")
print(veh[["label", "current_status", "speed_mph", "latitude", "longitude"]].head(3).to_string(index=False))

print("\n=== Predictions — Red Line inbound ===")
preds = api.get_predictions("Red", 1)
print(f"Predictions: {len(preds)}")
print(preds[["stop_name", "delay_min", "status"]].head(5).to_string(index=False))

print("\n=== Live metrics ===")
metrics = api.compute_live_metrics(preds)
for k, v in metrics.items():
    print(f"  {k}: {v}")

print("\n=== Alerts — Red Line ===")
alerts = api.get_alerts("Red")
print(f"Active alerts: {len(alerts)}")
for a in alerts:
    print(f"  [{a['severity']}] {a['effect']}: {a['header'][:60]}")




=== Vehicles — Red Line inbound ===
Active vehicles: 10
label current_status speed_mph  latitude  longitude
 1835    Incoming At      None  42.38416  -71.11936
 1756     Stopped At      None  42.27582  -71.03012
 1719    Incoming At      None  42.30029  -71.05826

=== Predictions — Red Line inbound ===
Predictions: 200
stop_name  delay_min status
Braintree        1.0       
Braintree        7.2       
Braintree        1.7       
Braintree        3.3       
Braintree       10.7       

=== Live metrics ===
  otp_pct: 50.0
  avg_delay_min: 6.2
  ett_min: 6.18
  on_time: 60
  late: 60
  early: 0
  total: 120

=== Alerts — Red Line ===
Active alerts: 0
